Import Library & Muat Dataset

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

app_cols = [
    'SK_ID_CURR', 'TARGET', 'NAME_CONTRACT_TYPE', 'CODE_GENDER',
    'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'CNT_CHILDREN', 'CNT_FAM_MEMBERS',
    'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE',
    'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS',
    'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE', 'ORGANIZATION_TYPE',
    'DAYS_BIRTH', 'DAYS_EMPLOYED', 'DAYS_REGISTRATION',
    'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3'
]

bureau_cols = [
    'SK_ID_CURR', 'SK_ID_BUREAU', 'CREDIT_ACTIVE', 'CREDIT_TYPE',
    'DAYS_CREDIT', 'CREDIT_DAY_OVERDUE', 'AMT_CREDIT_SUM', 'AMT_CREDIT_SUM_DEBT'
]

prev_cols = [
    'SK_ID_CURR', 'SK_ID_PREV', 'NAME_CONTRACT_TYPE', 'NAME_CONTRACT_STATUS',
    'AMT_APPLICATION', 'AMT_CREDIT', 'DAYS_DECISION', 'NAME_CLIENT_TYPE'
]

df_app = pd.read_csv('../data/raw/HC_application_train.csv', usecols=app_cols)
df_bureau = pd.read_csv('../data/raw/HC_bureau.csv', usecols=bureau_cols)
df_prev = pd.read_csv('../data/raw/HC_previous_application.csv', usecols=prev_cols)

print("Dataset berhasil dimuat!")
print(f"1. Application Train    : {df_app.shape[0]:,} baris x {df_app.shape[1]} kolom")
print(f"2. Bureau History       : {df_bureau.shape[0]:,} baris x {df_bureau.shape[1]} kolom")
print(f"3. Previous Application : {df_prev.shape[0]:,} baris x {df_prev.shape[1]} kolom")

Dataset berhasil dimuat!
1. Application Train    : 307,511 baris x 24 kolom
2. Bureau History       : 1,716,428 baris x 8 kolom
3. Previous Application : 1,670,214 baris x 8 kolom


Audit Kunci Relasi Antar-Tabel

In [ ]:
print("=== 1. AUDIT PRIMARY KEY & RELATIONAL COVERAGE ===")

unique_app = df_app['SK_ID_CURR'].nunique()
print(f"Total baris df_app       : {len(df_app):,}")
print(f"Unique SK_ID_CURR df_app : {unique_app:,} (Duplikat PK: {len(df_app) - unique_app})")

app_ids = set(df_app['SK_ID_CURR'])
bureau_ids = set(df_bureau['SK_ID_CURR'])
prev_ids = set(df_prev['SK_ID_CURR'])

match_bureau = len(app_ids.intersection(bureau_ids))
match_prev = len(app_ids.intersection(prev_ids))

print(f"\nCakupan Relasi (Persiapan LEFT JOIN SQL):")
print(f"- Peminjam punya riwayat di Bureau       : {match_bureau:,} ({match_bureau/unique_app*100:.2f}%)")
print(f"- Peminjam punya riwayat di Prev App     : {match_prev:,} ({match_prev/unique_app*100:.2f}%)")

print("\n=== 2. CONTOH 5 BARIS PERTAMA (APPLICATION TRAIN) ===")
display(df_app.head())

=== 1. AUDIT PRIMARY KEY & RELATIONAL COVERAGE ===
Total baris df_app       : 307,511
Unique SK_ID_CURR df_app : 307,511 (Duplikat PK: 0)

Cakupan Relasi (Persiapan LEFT JOIN SQL):
- Peminjam punya riwayat di Bureau       : 263,491 (85.69%)
- Peminjam punya riwayat di Prev App     : 291,057 (94.65%)

=== 2. CONTOH 5 BARIS PERTAMA (APPLICATION TRAIN) ===


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,DAYS_BIRTH,DAYS_EMPLOYED,DAYS_REGISTRATION,OCCUPATION_TYPE,CNT_FAM_MEMBERS,ORGANIZATION_TYPE,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3
0,100002,1,Cash loans,M,N,Y,0,"202,500.00","406,597.50","24,700.50","351,000.00",Working,Secondary / secondary special,Single / not married,House / apartment,-9461,-637,"-3,648.00",Laborers,1.00,Business Entity Type 3,0.08,0.26,0.14
1,100003,0,Cash loans,F,N,N,0,"270,000.00","1,293,502.50","35,698.50","1,129,500.00",State servant,Higher education,Married,House / apartment,-16765,-1188,"-1,186.00",Core staff,2.00,School,0.31,0.62,NaN
2,100004,0,Revolving loans,M,Y,Y,0,"67,500.00","135,000.00","6,750.00","135,000.00",Working,Secondary / secondary special,Single / not married,House / apartment,-19046,-225,"-4,260.00",Laborers,1.00,Government,NaN,0.56,0.73
3,100006,0,Cash loans,F,N,Y,0,"135,000.00","312,682.50","29,686.50","297,000.00",Working,Secondary / secondary special,Civil marriage,House / apartment,-19005,-3039,"-9,833.00",Laborers,2.00,Business Entity Type 3,NaN,0.65,NaN
4,100007,0,Cash loans,M,N,Y,0,"121,500.00","513,000.00","21,865.50","513,000.00",Working,Secondary / secondary special,Single / not married,House / apartment,-19932,-3038,"-4,311.00",Core staff,1.00,Religion,NaN,0.32,NaN


Audit Distribusi Target

In [3]:
print("=== AUDIT DISTRIBUSI TARGET (LOAN STATUS) ===")

target_counts = df_app['TARGET'].value_counts()
target_pct = df_app['TARGET'].value_counts(normalize=True) * 100

df_target_summary = pd.DataFrame({
    'Status_Label': ['0 - Lancar (Repaid)', '1 - Gagal Bayar (Default)'],
    'Jumlah_Nasabah': target_counts.values,
    'Persentase (%)': target_pct.values.round(2)
})

display(df_target_summary)

=== AUDIT DISTRIBUSI TARGET (LOAN STATUS) ===


,Status_Label,Jumlah_Nasabah,Persentase (%)
0,0 - Lancar (Repaid),282686,91.93
1,1 - Gagal Bayar (Default),24825,8.07


Audit Missing Values, Tipe Data, & Duplikat di Ketiga Tabel

In [ ]:
def audit_data_quality(df, table_name):
    print(f"\n{'='*60}")
    print(f"AUDIT KUALITAS DATA: {table_name}")
    print(f"{'='*60}")
    print(f"Jumlah Baris Duplikat Penuh: {df.duplicated().sum():,}")
    
    audit_df = pd.DataFrame({
        'Tipe_Data': df.dtypes,
        'Missing_Count': df.isnull().sum(),
        'Missing_Pct (%)': (df.isnull().sum() / len(df) * 100).round(2),
        'Unique_Values': df.nunique()
    })
    
    audit_df = audit_df.sort_values(by='Missing_Pct (%)', ascending=False)
    display(audit_df)

audit_data_quality(df_app, "HC_application_train (Selected Columns)")
audit_data_quality(df_bureau, "HC_bureau (Selected Columns)")
audit_data_quality(df_prev, "HC_previous_application (Selected Columns)")


AUDIT KUALITAS DATA: HC_application_train (Selected Columns)
Jumlah Baris Duplikat Penuh: 0


,Tipe_Data,Missing_Count,Missing_Pct (%),Unique_Values
EXT_SOURCE_1,float64,173378,56.38,114584
OCCUPATION_TYPE,str,96391,31.35,18
EXT_SOURCE_3,float64,60965,19.83,814
EXT_SOURCE_2,float64,660,0.21,119831
AMT_GOODS_PRICE,float64,278,0.09,1002
SK_ID_CURR,int64,0,0.00,307511
NAME_CONTRACT_TYPE,str,0,0.00,2
TARGET,int64,0,0.00,2
AMT_INCOME_TOTAL,float64,0,0.00,2548
CNT_CHILDREN,int64,0,0.00,15



AUDIT KUALITAS DATA: HC_bureau (Selected Columns)
Jumlah Baris Duplikat Penuh: 0


,Tipe_Data,Missing_Count,Missing_Pct (%),Unique_Values
AMT_CREDIT_SUM_DEBT,float64,257669,15.01,226537
SK_ID_CURR,int64,0,0.00,305811
CREDIT_ACTIVE,str,0,0.00,4
SK_ID_BUREAU,int64,0,0.00,1716428
DAYS_CREDIT,int64,0,0.00,2923
CREDIT_DAY_OVERDUE,int64,0,0.00,942
AMT_CREDIT_SUM,float64,13,0.00,236708
CREDIT_TYPE,str,0,0.00,15



AUDIT KUALITAS DATA: HC_previous_application (Selected Columns)
Jumlah Baris Duplikat Penuh: 0


,Tipe_Data,Missing_Count,Missing_Pct (%),Unique_Values
SK_ID_PREV,int64,0,0.00,1670214
SK_ID_CURR,int64,0,0.00,338857
NAME_CONTRACT_TYPE,str,0,0.00,4
AMT_APPLICATION,float64,0,0.00,93885
AMT_CREDIT,float64,1,0.00,86803
NAME_CONTRACT_STATUS,str,0,0.00,4
DAYS_DECISION,int64,0,0.00,2922
NAME_CLIENT_TYPE,str,0,0.00,4


Audit Anomali Domain Perbankan & Statistik Deskriptif

In [ ]:
print("=== 1. STATISTIK DESKRIPTIF VARIABEL NUMERIK (df_app) ===")
display(df_app.describe().T[['count', 'mean', '50%', 'min', 'max']])

print("\n=== 2. DETEKSI ANOMALI SPESIFIK DOMAIN KREDIT ===")
anom_emp = (df_app['DAYS_EMPLOYED'] == 365243).sum()
print(f"- Baris dengan DAYS_EMPLOYED == 365243 : {anom_emp:,} ({anom_emp/len(df_app)*100:.2f}%)")

pos_birth = (df_app['DAYS_BIRTH'] > 0).sum()
pos_emp = ((df_app['DAYS_EMPLOYED'] > 0) & (df_app['DAYS_EMPLOYED'] != 365243)).sum()
print(f"- Baris dengan DAYS_BIRTH > 0          : {pos_birth}")
print(f"- Baris dengan DAYS_EMPLOYED > 0 (lain): {pos_emp}")

zero_inc = (df_app['AMT_INCOME_TOTAL'] <= 0).sum()
zero_cred = (df_app['AMT_CREDIT'] <= 0).sum()
print(f"- Pendapatan <= 0                      : {zero_inc}")
print(f"- Nominal Kredit <= 0                  : {zero_cred}")

print("\nDistribusi CODE_GENDER:")
print(df_app['CODE_GENDER'].value_counts(dropna=False))

print("\nDistribusi NAME_INCOME_TYPE:")
print(df_app['NAME_INCOME_TYPE'].value_counts(dropna=False))

neg_debt = (df_bureau['AMT_CREDIT_SUM_DEBT'] < 0).sum()
print(f"\n- Sisa utang negatif di df_bureau (AMT_CREDIT_SUM_DEBT < 0): {neg_debt:,}")

=== 1. STATISTIK DESKRIPTIF VARIABEL NUMERIK (df_app) ===


,count,mean,50%,min,max
SK_ID_CURR,"307,511.00","278,180.52","278,202.00","100,002.00","456,255.00"
TARGET,"307,511.00",0.08,0.00,0.00,1.00
CNT_CHILDREN,"307,511.00",0.42,0.00,0.00,19.00
AMT_INCOME_TOTAL,"307,511.00","168,797.92","147,150.00","25,650.00","117,000,000.00"
AMT_CREDIT,"307,511.00","599,026.00","513,531.00","45,000.00","4,050,000.00"
AMT_ANNUITY,"307,499.00","27,108.57","24,903.00","1,615.50","258,025.50"
AMT_GOODS_PRICE,"307,233.00","538,396.21","450,000.00","40,500.00","4,050,000.00"
DAYS_BIRTH,"307,511.00","-16,037.00","-15,750.00","-25,229.00","-7,489.00"
DAYS_EMPLOYED,"307,511.00","63,815.05","-1,213.00","-17,912.00","365,243.00"
DAYS_REGISTRATION,"307,511.00","-4,986.12","-4,504.00","-24,672.00",0.00



=== 2. DETEKSI ANOMALI SPESIFIK DOMAIN KREDIT ===
- Baris dengan DAYS_EMPLOYED == 365243 : 55,374 (18.01%)
- Baris dengan DAYS_BIRTH > 0          : 0
- Baris dengan DAYS_EMPLOYED > 0 (lain): 0
- Pendapatan <= 0                      : 0
- Nominal Kredit <= 0                  : 0

Distribusi CODE_GENDER:
CODE_GENDER
F      202448
M      105059
XNA         4
Name: count, dtype: int64

Distribusi NAME_INCOME_TYPE:
NAME_INCOME_TYPE
Working                 158774
Commercial associate     71617
Pensioner                55362
State servant            21703
Unemployed                  22
Student                     18
Businessman                 10
Maternity leave              5
Name: count, dtype: int64

- Sisa utang negatif di df_bureau (AMT_CREDIT_SUM_DEBT < 0): 8,418


Kamus Referensi Kolom Resm

In [ ]:
desc_df = pd.read_csv('../data/raw/HomeCredit_columns_description.csv', encoding='latin1')

app_desc = desc_df[
    (desc_df['Table'] == 'application_{train|test}.csv') & 
    (desc_df['Row'].isin(app_cols))
][['Row', 'Description', 'Special']]

print("=== KAMUS DESKRIPSI KOLOM TERPILIH (APPLICATION TRAIN) ===")
for _, row in app_desc.iterrows():
    print(f"- {row['Row']}: {row['Description']}")

=== KAMUS DESKRIPSI KOLOM TERPILIH (APPLICATION TRAIN) ===
- SK_ID_CURR: ID of loan in our sample
- TARGET: Target variable (1 - client with payment difficulties: he/she had late payment more than X days on at least one of the first Y installments of the loan in our sample, 0 - all other cases)
- NAME_CONTRACT_TYPE: Identification if loan is cash or revolving
- CODE_GENDER: Gender of the client
- FLAG_OWN_CAR: Flag if the client owns a car
- FLAG_OWN_REALTY: Flag if client owns a house or flat
- CNT_CHILDREN: Number of children the client has
- AMT_INCOME_TOTAL: Income of the client
- AMT_CREDIT: Credit amount of the loan
- AMT_ANNUITY: Loan annuity
- AMT_GOODS_PRICE: For consumer loans it is the price of the goods for which the loan is given
- NAME_INCOME_TYPE: Clients income type (businessman, working, maternity leave,)
- NAME_EDUCATION_TYPE: Level of highest education the client achieved
- NAME_FAMILY_STATUS: Family status of the client
- NAME_HOUSING_TYPE: What is the housing situ